In [1]:
import json

id2label = {
    0: "B-LOC",
    1: "B-MISC",
    2: "B-ORG",
    3: "B-PER",
    4: "I-LOC",
    5: "I-MISC",
    6: "I-ORG",
    7: "I-PER",
    8: "O"
}

def load(path):
    return [json.loads(line) for line in open(path, "r", encoding="utf8")]

def extract_spans(tokens, tags):
    spans = []
    current = None

    for i, tag_id in enumerate(tags):
        tag = id2label[tag_id]

        if tag == "O":
            if current:
                spans.append(current)
                current = None
            continue

        prefix, label = tag.split("-")

        if prefix == "B":
            if current:
                spans.append(current)
            current = {"label": label, "start": i, "end": i+1}

        elif prefix == "I":
            if current and current["label"] == label:
                current["end"] = i+1
            else:
                # I-X without matching B-X → treat as new entity
                current = {"label": label, "start": i, "end": i+1}

    if current:
        spans.append(current)

    # Convert to readable form
    readable = []
    for s in spans:
        text = " ".join(tokens[s["start"]:s["end"]])
        readable.append((s["label"], text))
    return readable

def analyze(a1, a2, a3):
    A = load(a1)
    B = load(a2)
    C = load(a3)

    for idx, (x, y, z) in enumerate(zip(A, B, C)):
        spansA = extract_spans(x["tokens"], x["tags"])
        spansB = extract_spans(y["tokens"], y["tags"])
        spansC = extract_spans(z["tokens"], z["tags"])

        if not (spansA == spansB == spansC):
            print("="*80)
            print(f"TWEET {idx}")
            print("TOKENS:", " ".join(x["tokens"]))
            print()
            print("Annotator A:", spansA)
            print("Annotator B:", spansB)
            print("Annotator C:", spansC)
            print()

# Example:
analyze("val_500.jsonl", "Hjalte_500.jsonl", "mar_500.jsonl")


TWEET 6
TOKENS: RT @USER445 : Spine - Chilling Stories of Real Life Cannibals , Horrible ! URL1310 URL1455

Annotator A: []
Annotator B: []
Annotator C: [('MISC', 'Spine')]

TWEET 7
TOKENS: I 've entered for a chance to win a copy of @USER1136 on XB1 from @USER1071 thanks to @USER718 ! Enter HERE : URL1617

Annotator A: [('MISC', 'XB1')]
Annotator B: [('ORG', 'XB1')]
Annotator C: [('ORG', 'XB1')]

TWEET 11
TOKENS: RT @USER487 : Where is AKMU Where is MADE full album Where is EXIT : X Where is YGGG Saddest ? Where is Minzy ?

Annotator A: [('MISC', 'AKMU'), ('MISC', 'MADE'), ('MISC', 'EXIT'), ('MISC', 'YGGG Saddest'), ('PER', 'Minzy')]
Annotator B: [('MISC', 'AKMU'), ('MISC', 'MADE'), ('MISC', 'EXIT'), ('MISC', 'YGGG'), ('PER', 'Minzy')]
Annotator C: [('PER', 'AKMU'), ('PER', 'Minzy')]

TWEET 12
TOKENS: RT @USER2437 : If you have short memory let me inform you after signing COD Benazir made infamous NRO deal with Mushi . ☺ URL224 …

Annotator A: [('MISC', 'COD'), ('PER', 'Benazir'), ('M

In [2]:
def count_disagreements(a1, a2, a3):
    A = load(a1)
    B = load(a2)
    C = load(a3)

    total_tweets = len(A)
    disagree_tweets = 0
    total_tokens = 0
    tokens_labeled_by_someone = 0
    token_disagreements = 0

    for x, y, z in zip(A, B, C):
        spansA = extract_spans(x["tokens"], x["tags"])
        spansB = extract_spans(y["tokens"], y["tags"])
        spansC = extract_spans(z["tokens"], z["tags"])

        # Count tokens
        total_tokens += len(x["tokens"])

        # Count tokens where at least one annotator labeled something ≠ O (8)
        for tA, tB, tC in zip(x["tags"], y["tags"], z["tags"]):
            if not (tA == 8 and tB == 8 and tC == 8):
                tokens_labeled_by_someone += 1

        # Tweet-level disagreement
        if not (spansA == spansB == spansC):
            disagree_tweets += 1

            # Token-level disagreement count
            for tA, tB, tC in zip(x["tags"], y["tags"], z["tags"]):
                if not (tA == tB == tC):
                    token_disagreements += 1

    print("=== DISAGREEMENT SUMMARY ===")
    print(f"Total tweets: {total_tweets}")
    print(f"Tweets with disagreement: {disagree_tweets}")
    print(f"Tweets with full agreement: {total_tweets - disagree_tweets}")
    print()

    print(f"Tokens labeled by at least one annotator: {tokens_labeled_by_someone}")
    print(f"Total tokens: {total_tokens}")
    print()

    print(f"Tokens with disagreement: {token_disagreements}")
    print(f"Agreement rate (tokens): {(1 - token_disagreements/total_tokens):.4f}")
    print(f"Disagreement rate (tokens): {(token_disagreements/total_tokens):.4f}")

# Example usage:
count_disagreements("val_500.jsonl", "Hjalte_500.jsonl", "mar_500.jsonl")


=== DISAGREEMENT SUMMARY ===
Total tweets: 500
Tweets with disagreement: 154
Tweets with full agreement: 346

Tokens labeled by at least one annotator: 545
Total tokens: 7658

Tokens with disagreement: 361
Agreement rate (tokens): 0.9529
Disagreement rate (tokens): 0.0471
